# 05B — GMM : segmentation probabiliste | Panier Vert

Chaque notebook lit **le même** `data/gold/user_segments.parquet` produit par le notebook 04. Les transformations (`log1p`, normalisation) et l'échantillon pseudo-aléatoire de 15 000 clients sont identiques : la comparaison est cohérente. Les résultats sont tracés dans `mlruns/` à la racine du projet ; aucune instance MLflow ne doit être lancée pour exécuter les notebooks. Pour afficher l'interface, depuis la racine : `mlflow ui --backend-store-uri ./mlruns`.

**Note méthodologique :** la normalisation est ajustée ici sur l'ensemble des clients avant l'échantillonnage, comme dans le notebook initial. Pour une évaluation hors échantillon stricte, il faudrait ajuster les transformations sur un jeu d'entraînement seulement. Les métriques internes de clustering ne garantissent pas à elles seules une segmentation utile métier.

## 1 — Chargement et préparation

In [ ]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

cwd = Path.cwd()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "data" / "gold" / "user_segments.parquet").exists()), None)
if ROOT is None:
    raise FileNotFoundError("data/gold/user_segments.parquet introuvable : exécuter le notebook 04 ou adapter ROOT.")
GOLD = ROOT / "data" / "gold" / "user_segments.parquet"
REPORT_DIR = ROOT / "data" / "reports" / "segmentation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri((ROOT / "mlruns").resolve().as_uri())

df = pd.read_parquet(GOLD).set_index("user_id")
print(f"{len(df):,} clients | {df.shape[1]} colonnes")

# Préparation identique pour les trois modèles
DROP = ["tenure_days", "nb_distinct_products", "product_diversity", "segment", "segment_name"]
feat = df.drop(columns=[c for c in DROP if c in df.columns]).select_dtypes(include=["number"]).copy()
if feat.empty:
    raise ValueError("Aucune variable numérique disponible après exclusion.")
if feat.isna().any().any():
    raise ValueError("Valeurs manquantes dans les variables : vérifier le notebook 04.")

X = feat.copy()
skewed = [c for c in X.columns if X[c].skew() > 1 and (X[c] >= 0).all()]
X[skewed] = np.log1p(X[skewed])
scaler = StandardScaler()
Z = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)
K = int(df["segment"].nunique()) if "segment" in df.columns else 4
CMP_N = 15_000
rng = np.random.RandomState(RANDOM_STATE)
cmp_idx = rng.choice(len(Z), size=min(CMP_N, len(Z)), replace=False)
Zc = Z.iloc[cmp_idx].to_numpy()
print("Variables :", list(Z.columns))
print("Échantillon commun :", Zc.shape, "| K de référence :", K)

def segmentation_metrics(data, labels, *, exclude_noise=True):
    labels = np.asarray(labels)
    mask = (labels != -1) if exclude_noise else np.ones(len(labels), dtype=bool)
    unique = np.unique(labels[mask])
    out = {"n_clusters": int(len(unique)), "noise_pct": float(100 * (~mask).mean()),
           "classified_pct": float(100 * mask.mean())}
    if len(unique) >= 2 and mask.sum() > len(unique):
        # Silhouette sur un sous-échantillon pour éviter la matrice O(n²)
        out["silhouette"] = float(silhouette_score(data[mask], labels[mask],
            sample_size=min(2500, int(mask.sum())), random_state=RANDOM_STATE))
        out["davies_bouldin"] = float(davies_bouldin_score(data[mask], labels[mask]))
    else:
        out["silhouette"] = float("nan")
        out["davies_bouldin"] = float("nan")
    if mask.any():
        out["largest_cluster_pct"] = float(100 * pd.Series(labels[mask]).value_counts(normalize=True).iloc[0])
    return out

def log_numeric_metrics(d):
    for k, v in d.items():
        if isinstance(v, (int, float, np.integer, np.floating)) and np.isfinite(v):
            mlflow.log_metric(k, float(v))


## 2 — BIC pour évaluer le nombre de composantes

In [ ]:
from sklearn.mixture import GaussianMixture
mlflow.set_experiment("PanierVert_GMM_Clients")
BIC_N = 30_000
bic_idx = rng.choice(len(Z), size=min(BIC_N, len(Z)), replace=False)
Zb = Z.iloc[bic_idx].to_numpy()

bic_rows = []
for k in range(2, 9):
    with mlflow.start_run(run_name=f"gmm_bic_k{k}"):
        t0 = time.perf_counter()
        model_bic = GaussianMixture(n_components=k, covariance_type="full",
                                    n_init=1, random_state=RANDOM_STATE).fit(Zb)
        duration = time.perf_counter() - t0
        row = {"k": k, "BIC": model_bic.bic(Zb), "AIC": model_bic.aic(Zb), "duration_s": duration}
        bic_rows.append(row)
        mlflow.log_params({"model": "GMM", "n_components": k, "covariance_type": "full",
                           "sample_size": len(Zb)})
        log_numeric_metrics(row)
gmm_sel = pd.DataFrame(bic_rows).set_index("k")
display(gmm_sel)
ax = gmm_sel[["BIC", "AIC"]].plot(marker="o", title="GMM : BIC / AIC selon K")
plt.tight_layout(); plt.savefig(REPORT_DIR / "gmm_bic.png", dpi=130); plt.show()
print("K minimum BIC :", int(gmm_sel.BIC.idxmin()), "| K fixé pour comparaison :", K)

## 3 — GMM sur l’échantillon commun et suivi MLflow

In [ ]:
with mlflow.start_run(run_name=f"gmm_compare_k{K}_n{len(Zc)}"):
    t0 = time.perf_counter()
    model = GaussianMixture(n_components=K, covariance_type="full", n_init=3,
                            random_state=RANDOM_STATE).fit(Zc)
    labels = model.predict(Zc)
    confidence = model.predict_proba(Zc).max(axis=1)
    scores = segmentation_metrics(Zc, labels)
    scores.update({"duration_s": time.perf_counter() - t0,
                   "mean_assignment_confidence": float(confidence.mean()),
                   "ambiguous_pct": float(100 * (confidence < 0.6).mean())})
    mlflow.log_params({"model": "GMM", "n_components": K, "covariance_type": "full",
                      "sample_size": len(Zc), "random_state": RANDOM_STATE})
    log_numeric_metrics(scores)
    display(pd.DataFrame([scores]))
    profile = feat.iloc[cmp_idx].groupby(labels).mean().round(2)
    display(profile)
    profile.to_csv(REPORT_DIR / "profil_gmm_15k.csv")
    mlflow.log_artifact(str(REPORT_DIR / "profil_gmm_15k.csv"))
    fig, ax = plt.subplots(figsize=(7, 5))
    xy = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Zc)
    ax.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="tab10", s=6, alpha=.65)
    ax.set(title="GMM — projection PCA", xlabel="PC1", ylabel="PC2")
    fig.tight_layout(); fig.savefig(REPORT_DIR / "gmm_pca.png", dpi=130)
    mlflow.log_artifact(str(REPORT_DIR / "gmm_pca.png"))
    plt.show()

## 4 — Export complet (optionnel)

Le modèle initial entraînait également un GMM sur **tous** les clients. Ce calcul est coûteux : exécute la cellule suivante uniquement si tu souhaites produire la table Gold complémentaire.

In [ ]:
RUN_FULL_GMM = False  # Passer à True pour entraîner sur tous les clients
if RUN_FULL_GMM:
    with mlflow.start_run(run_name=f"gmm_full_k{K}"):
        t0 = time.perf_counter()
        model_full = GaussianMixture(n_components=K, covariance_type="full",
                                     n_init=3, random_state=RANDOM_STATE).fit(Z.to_numpy())
        proba = model_full.predict_proba(Z.to_numpy())
        seg_gmm = proba.argmax(axis=1)
        conf = proba.max(axis=1)
        out = df.copy()
        out["segment_gmm"] = seg_gmm
        out["gmm_confiance"] = conf.round(4)
        out_path = ROOT / "data" / "gold" / "user_segments_compare.parquet"
        out.reset_index().to_parquet(out_path, index=False)
        mlflow.log_params({"model": "GMM", "n_components": K,
                           "sample_size": len(Z), "covariance_type": "full"})
        mlflow.log_metrics({"mean_assignment_confidence": float(conf.mean()),
                            "ambiguous_pct": float(100 * (conf < .6).mean()),
                            "duration_s": time.perf_counter() - t0})
        print("Écrit :", out_path)
else:
    print("Export complet désactivé.")